# GraviText - Tokenization, embeddings and transformers

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "../src/GraviFrame/bin/Release/net10.0/Gravicode.Science.GraviFrame.dll"
#r "../src/GraviLearn/bin/Release/net10.0/Gravicode.Science.GraviLearn.dll"
#r "../src/GraviText/bin/Release/net10.0/Gravicode.Science.GraviText.dll"
#r "nuget: ScottPlot, 5.1.59"

using Gravicode.Science.GraviFrame;
using Gravicode.Science.GraviLearn.Decomposition;
using Gravicode.Science.GraviNum;
using Gravicode.Science.GraviText.Embeddings;
using Gravicode.Science.GraviText.Linguistics;
using Gravicode.Science.GraviText.Tasks;
using Gravicode.Science.GraviText.Tokenization;
using Gravicode.Science.GraviText.Transformers;
using Gravicode.Science.GraviText.Vectorization;

var reviews = DataFrame.ReadCsv("../datasets/imdb_reviews.csv");
var documents = Enumerable.Range(0, reviews.RowCount).Select(i => reviews.Text("review")[i]!).ToArray();
var labels = Enumerable.Range(0, reviews.RowCount).Select(i => reviews.Text("sentiment")[i]!).ToArray();
Console.WriteLine($"{documents.Length} reviews");

## Tokenization and stemming, in both languages

In [ ]:
Console.WriteLine(string.Join(" | ", new RegexTokenizer().Tokenize("Gravicode Studios membangun AI di .NET!")));

foreach (var w in new[] { "connection", "running", "ponies" })
    Console.WriteLine($"en {w,-14}-> {PorterStemmer.Stem(w)}");
foreach (var w in new[] { "makanan", "membaca", "berlari" })
    Console.WriteLine($"id {w,-14}-> {IndonesianStemmer.Stem(w)}");

## Sentiment analysis

In [ ]:
var rng = new GraviRandom(42);
var order = rng.Permutation(documents.Length);
var cut = (int)(documents.Length * 0.75);

var trainDocs = order.Take(cut).Select(i => documents[i]).ToArray();
var trainLabels = order.Take(cut).Select(i => labels[i]).ToArray();
var testDocs = order.Skip(cut).Select(i => documents[i]).ToArray();
var testLabels = order.Skip(cut).Select(i => labels[i]).ToArray();

var classifier = new TextClassifier().Train(trainDocs, trainLabels);
Console.WriteLine($"held-out accuracy: {classifier.Evaluate(testDocs, testLabels):P2}");

foreach (var label in classifier.Labels)
    Console.WriteLine($"{label,-10}{string.Join(", ", classifier.TopFeatures(label, 8).Select(t => t.Term))}");

## Word embeddings and a t-SNE projection

Raw vectors share a large common direction; removing it is what makes the cosine informative on a small corpus.

In [ ]:
var tokenized = documents.Select(d => new RegexTokenizer().Tokenize(d)).ToList();
var raw = new Word2Vec(dimensions: 64, windowSize: 4, minCount: 2, epochs: 25, seed: 42).Train(tokenized);
var embeddings = raw.RemoveCommonComponent();

var probe = raw.Words.Take(40).ToArray();
Console.WriteLine($"mean pairwise cosine: {probe.SelectMany(a => probe.Where(b => b != a).Select(b => raw.Similarity(a, b))).Average():F3} raw");
Console.WriteLine($"                      {probe.SelectMany(a => probe.Where(b => b != a).Select(b => embeddings.Similarity(a, b))).Average():F3} centred");

In [ ]:
var vocabulary = embeddings.Words.Where(w => w.Length > 3).Take(90).ToArray();
var vectors = NdArray.Zeros(vocabulary.Length, embeddings.Dimensions);
for (var i = 0; i < vocabulary.Length; i++)
{
    var v = embeddings[vocabulary[i]];
    for (var d = 0; d < embeddings.Dimensions; d++) vectors[i, d] = v.At(d);
}

var projection = new TStochasticNeighborEmbedding(2, perplexity: 12, iterations: 400, seed: 42).FitTransform(vectors);

var plot = new ScottPlot.Plot();
var xs = Enumerable.Range(0, vocabulary.Length).Select(i => projection[i, 0]).ToArray();
var ys = Enumerable.Range(0, vocabulary.Length).Select(i => projection[i, 1]).ToArray();
var scatter = plot.Add.ScatterPoints(xs, ys);
scatter.MarkerSize = 8;
for (var i = 0; i < vocabulary.Length; i += 3)
    plot.Add.Text(vocabulary[i], xs[i], ys[i]).LabelFontSize = 9;

plot.Title("Word vectors projected with t-SNE");
plot.GetImageHtml(950, 750)

## Transformer encoder

**No pretrained weights are bundled.** The forward pass is complete and correct, but a fresh model is randomly initialised, so its vectors are structurally valid rather than semantically meaningful. Call LoadWeights before treating the output as embeddings.

In [ ]:
var vocabulary2 = WordPieceTokenizer.Train(documents, vocabularySize: 900, minFrequency: 2);
var tokenizer = new WordPieceTokenizer(vocabulary2);
var config = new TransformerConfig(vocabulary2.Count, HiddenSize: 128, Layers: 4, Heads: 8, IntermediateSize: 512, MaxPositions: 128);
var model = new TransformerModel(config, seed: 42).WithTokenizer(tokenizer);

Console.WriteLine(model);
model.Forward(new[] { 1, 2, 3, 4, 5, 6 });

var attention = model.AttentionMaps[0][0];
var plot3 = new ScottPlot.Plot();
plot3.Add.Heatmap(attention.To2DArray());
plot3.Title("Layer 1, head 1 attention (rows sum to 1)");
plot3.GetImageHtml(600, 500)